# T02 — Reconciliação Financeira e Consistência dos Fluxos

**FRTB | Validação independente — etapa exploratória**  
**Data-base de referência:** 30/09/2026 (a confirmar pela rastreabilidade das extrações)

## 1. Objetivo, hipóteses e limites

Avaliar a **granularidade**, a **correspondência dos registros** e a **consistência financeira** entre as bases `fluxos` e `fluxos por cenario`.

- **H1:** há correspondência entre os fluxos convencionais e os registros comparáveis do cenário-base.
- **H2:** atributos financeiros semanticamente equivalentes são reconciliáveis dentro de tolerâncias definidas.
- **H3:** divergências podem ser quantificadas e investigadas sem multiplicação artificial de valores.

**Limites:** as duas bases podem compartilhar a mesma origem de cálculo; reconciliação interna **não é** apreçamento independente. A base `fluxos` não tem `DtPos`. O campo `CodComponente` deve ser investigado antes de tratar `Base` como cenário de referência. **Não interpretar a diferença de 382 linhas entre arquivos como reconciliação demonstrada.**

**Política de execução:** as células de perfil e granularidade são exploratórias. As comparações monetárias só são executadas quando a chave escolhida é única em ambas as populações comparadas. Não alterar bases de origem.

## 2. Configuração e carregamento

Estrutura de pastas:

```text
FRTB/
├── dados/
│   ├── fluxos.xlsx
│   └── fluxos por cenario.xlsx
└── Valid_FRTB/
    └── T02_Reconciliacao_Financeira.ipynb
```

Execute o notebook a partir de `Valid_FRTB`. Se necessário, altere `PASTA_DADOS`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

PASTA_ATUAL = Path.cwd().resolve()
PASTA_DADOS = (PASTA_ATUAL.parent / "dados") if PASTA_ATUAL.name.lower() == "valid_frtb" else (PASTA_ATUAL / "dados")
PASTA_RESULTADOS = PASTA_ATUAL / "resultados"
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
ARQUIVOS = {"fluxos": "fluxos.xlsx", "cenarios": "fluxos por cenario.xlsx"}

for nome in ARQUIVOS.values():
    if not (PASTA_DADOS / nome).is_file():
        raise FileNotFoundError(f"Arquivo não localizado: {PASTA_DADOS / nome}")

flx = pd.read_excel(PASTA_DADOS / ARQUIVOS["fluxos"])
cen = pd.read_excel(PASTA_DADOS / ARQUIVOS["cenarios"])
for df in (flx, cen):
    df.columns = df.columns.astype(str).str.strip()
    if df.columns.duplicated().any():
        raise ValueError("Colunas duplicadas após remoção de espaços; revisar a origem.")
print("Fluxos:", flx.shape, "| Cenários:", cen.shape)
print("Pasta de evidências:", PASTA_RESULTADOS)

## 3. Perfil dos cenários

O agrupamento abaixo descreve as combinações observadas, **sem presumir** que todos os registros com `CodComponente = Base` são diretamente comparáveis aos fluxos convencionais.

In [ ]:
colunas_cenario = ["CodComponente", "CodClasse", "CodCategoria", "CodFatorSens", "VlrChoqueSens"]
colunas_disponiveis = [c for c in colunas_cenario if c in cen.columns]
if not colunas_disponiveis:
    raise ValueError("Nenhuma coluna de identificação de cenário foi encontrada.")
perfil_cenarios = (cen.groupby(colunas_disponiveis, dropna=False)
                   .size().reset_index(name="Qtd_Registros")
                   .sort_values("Qtd_Registros", ascending=False))
display(perfil_cenarios.head(30))
print("Total de registros no perfil:", perfil_cenarios.Qtd_Registros.sum())

## 4. Granularidade e chaves candidatas

As chaves são **candidatas**, não regras de negócio confirmadas. Uma chave com repetições em cenários pode ser legítima devido às dimensões adicionais. A quantidade de IDs repetidos não equivale à quantidade de duplicidades econômicas.

In [ ]:
chaves_candidatas = ["IDPos", "CodCompraVenda", "CodCategCont", "IdFluxo"]
chaves = [c for c in chaves_candidatas if c in flx.columns and c in cen.columns]
if "IDPos" not in chaves:
    raise ValueError("IDPos ausente em uma das bases.")

def diagnostico_chaves(df, nome, chave):
    return {"Base": nome, "Registros": len(df),
            "Chaves_distintas": len(df.drop_duplicates(chave)),
            "Linhas_com_chave_repetida": int(df.duplicated(chave, keep=False).sum()),
            "Linhas_com_chave_nula": int(df[chave].isna().any(axis=1).sum())}

diagnostico = pd.DataFrame([
    diagnostico_chaves(flx, "Fluxos", chaves),
    diagnostico_chaves(cen, "Cenarios (todos)", chaves)
])
print("Chaves candidatas compartilhadas:", chaves)
display(diagnostico)

## 5. População candidata de cenário-base e correspondência

**Decisão metodológica provisória:** selecionar `CodComponente = Base` (sem distinção de maiúsculas/minúsculas), se presente. Essa seleção precisa ser confirmada com a documentação funcional. A reconciliação a seguir é **por conjunto de chaves**, sem `merge` de linhas monetárias, para não inflar contagens.

In [ ]:
if "CodComponente" not in cen.columns:
    raise ValueError("CodComponente não está presente; conferir o esquema de cenários.")
base = cen.loc[cen["CodComponente"].astype("string").str.strip().str.casefold().eq("base").fillna(False)].copy()
print("Registros candidatos a Base:", len(base))
if base.empty:
    print("ATENÇÃO: nenhum registro Base identificado; não executar reconciliação monetária.")

# Conjuntos de chaves; evita multiplicação por relacionamento muitos-para-muitos.
chaves_f = flx[chaves].dropna().drop_duplicates()
chaves_b = base[chaves].dropna().drop_duplicates()
comp_f = chaves_f.merge(chaves_b.assign(_presente=1), on=chaves, how="left", validate="one_to_one")
comp_b = chaves_b.merge(chaves_f.assign(_presente=1), on=chaves, how="left", validate="one_to_one")
resumo_correspondencia = pd.DataFrame([
    {"Metrica": "Chaves únicas em fluxos", "Quantidade": len(chaves_f)},
    {"Metrica": "Chaves únicas em Base", "Quantidade": len(chaves_b)},
    {"Metrica": "Chaves de fluxos sem Base", "Quantidade": int(comp_f._presente.isna().sum())},
    {"Metrica": "Chaves de Base sem fluxos", "Quantidade": int(comp_b._presente.isna().sum())},
    {"Metrica": "Linhas com chave nula em fluxos", "Quantidade": int(flx[chaves].isna().any(axis=1).sum())},
    {"Metrica": "Linhas com chave nula em Base", "Quantidade": int(base[chaves].isna().any(axis=1).sum())},
])
display(resumo_correspondencia)

## 6. Comparação financeira condicionada à unicidade

Só serão comparados campos com correspondência nominal plausível. **Equivalência econômica ainda deve ser confirmada** (sinal, moeda, indexação, valor por fluxo versus posição). Se a chave não for única nas duas bases, o código **bloqueia a comparação** em vez de agregar valores arbitrariamente.

Tolerâncias **exploratórias**: absoluta de 0,01 e relativa de 1e-6, ajustáveis após conhecer unidades e regras de arredondamento.

In [ ]:
TOL_ABS = 0.01
TOL_REL = 1e-6
pares_campos = [
    ("flx_VlrFuturo", "flx_VlrFuturo"),
    ("flx_VlrPresente", "flx_VlrPresente"),
    ("flx_VlrMaM", "flx_vlrMaM"),
]
pares_disponiveis = [(a,b) for a,b in pares_campos if a in flx.columns and b in base.columns]
print("Campos candidatos a comparação:", pares_disponiveis)

unicidade_f = not flx.duplicated(chaves).any() and not flx[chaves].isna().any(axis=1).any()
unicidade_b = not base.duplicated(chaves).any() and not base[chaves].isna().any(axis=1).any()
comparacao_financeira = pd.DataFrame()
resumo_financeiro = pd.DataFrame()

if not (unicidade_f and unicidade_b):
    print("COMPARAÇÃO BLOQUEADA: chaves repetidas ou nulas em pelo menos uma base.")
    print("Próxima ação: confirmar granularidade e dimensões de cenário antes de comparar valores.")
elif not pares_disponiveis:
    print("COMPARAÇÃO BLOQUEADA: nenhum par de campos monetários encontrado.")
else:
    cols_f = chaves + [a for a,_ in pares_disponiveis]
    cols_b = chaves + [b for _,b in pares_disponiveis]
    comparacao_financeira = flx[cols_f].merge(base[cols_b], on=chaves, how="inner", suffixes=("_flx", "_base"), validate="one_to_one")
    resultados = []
    for a,b in pares_disponiveis:
        cf = f"{a}_flx" if a == b else a
        cb = f"{b}_base" if a == b else b
        x = pd.to_numeric(comparacao_financeira[cf], errors="coerce")
        y = pd.to_numeric(comparacao_financeira[cb], errors="coerce")
        valido = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)
        dif = (x-y).abs()
        diverge = valido & (dif > TOL_ABS + TOL_REL * np.maximum(x.abs(), y.abs()))
        comparacao_financeira[f"Dif_{a}"] = dif
        comparacao_financeira[f"Diverge_{a}"] = diverge
        resultados.append({"Campo_fluxos": a, "Campo_base": b,
                           "Pares_validos": int(valido.sum()),
                           "Pares_sem_comparacao": int((~valido).sum()),
                           "Divergencias": int(diverge.sum()),
                           "Maior_diferenca_absoluta": float(dif[valido].max()) if valido.any() else np.nan})
    resumo_financeiro = pd.DataFrame(resultados)
    display(resumo_financeiro)
    print("Atenção: correspondência nominal não comprova equivalência econômica dos campos.")

## 7. Exportação das evidências

As evidências agregadas podem ser armazenadas em pasta interna. O arquivo não contém listagens de IDs, mas ainda pode revelar informações operacionais e não deve ser publicado sem autorização.

In [ ]:
saida = PASTA_RESULTADOS / "T02_reconciliacao_financeira_resultados.xlsx"
with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    perfil_cenarios.to_excel(writer, sheet_name="Perfil_Cenarios", index=False)
    diagnostico.to_excel(writer, sheet_name="Granularidade", index=False)
    resumo_correspondencia.to_excel(writer, sheet_name="Correspondencia", index=False)
    if not resumo_financeiro.empty:
        resumo_financeiro.to_excel(writer, sheet_name="Resumo_Financeiro", index=False)
    pd.DataFrame([{"Campo": "Data-base informada", "Valor": "2026-09-30"},
                  {"Campo": "Cenário selecionado", "Valor": "CodComponente=Base (hipótese a confirmar)"},
                  {"Campo": "Chaves candidatas", "Valor": ", ".join(chaves)},
                  {"Campo": "Unicidade em fluxos", "Valor": str(unicidade_f)},
                  {"Campo": "Unicidade em Base", "Valor": str(unicidade_b)},
                  {"Campo": "Tolerância absoluta exploratória", "Valor": TOL_ABS},
                  {"Campo": "Tolerância relativa exploratória", "Valor": TOL_REL}]).to_excel(writer, sheet_name="Premissas", index=False)
print("Evidências salvas em:", saida)

## 8. Parecer técnico — preencher após execução

**Status preliminar: EM ANÁLISE.**

A etapa inicial do T02 caracterizou a composição dos cenários, a granularidade dos registros e a correspondência de chaves candidatas entre as bases. A possibilidade de reconciliação monetária depende da unicidade das chaves e da confirmação da semântica dos campos.

**Pontos para concluir:**

1. Confirmar com a documentação funcional se `CodComponente = Base` corresponde à população apropriada para a reconciliação.
2. Verificar a chave econômica completa e as eventuais dimensões adicionais de cenário.
3. Confirmar unidades, sinais e definições dos campos monetários.
4. Investigar diferenças e estabelecer tolerâncias justificadas.
5. Registrar a rastreabilidade das extrações e a limitação temporal de `fluxos` (sem `DtPos`).

**Conclusão:** não atribuir parecer satisfatório até que os itens acima sejam avaliados e as evidências quantitativas sustentem a conclusão. A reconciliação interna não substitui a replicação independente de apreçamento com curvas e convenções de mercado.